<a href="https://colab.research.google.com/github/stephenkeyy77/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2012/Lab_12_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/stephenkeyy77/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [5]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [6]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()

#Output the diagnostic matrix
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        20:06:12   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [7]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [9]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"], y_pred)

# print the result formatted cleanly as a US Dollar currency string
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [10]:
"""
Residual Forensics Dashboard
=============================
Interactive OLS diagnostic tool using Plotly Express.
Built on a statsmodels hedonic pricing regression example.

Author: Yunyu | Course: ECON 3916
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─────────────────────────────────────────────────────────────
# STEP 1 ▸ Synthetic hedonic dataset (replace with your own)
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
n = 500

# Hedonic features: square footage, bedrooms, age, transit score
sqft        = np.random.normal(1500, 400, n).clip(500, 4000)
bedrooms    = np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.20, 0.45, 0.25, 0.05])
age         = np.random.uniform(0, 60, n)
transit     = np.random.uniform(0, 100, n)

# True price with heteroscedastic noise: variance scales with sqft
#   → intentionally baked in so the dashboard has something to *detect*
noise_scale = 0.08 * sqft                          # fan-shape pattern
log_price = (
    12.0
    + 0.0004 * sqft
    + 0.08   * bedrooms
    - 0.005  * age
    + 0.003  * transit
    + np.random.normal(0, noise_scale / sqft.mean(), n)
)
price = np.exp(log_price)

df = pd.DataFrame({
    "price":    price,
    "sqft":     sqft,
    "bedrooms": bedrooms,
    "age":      age,
    "transit":  transit,
})

# ─────────────────────────────────────────────────────────────
# STEP 2 ▸ OLS estimation via statsmodels
# ─────────────────────────────────────────────────────────────
X = df[["sqft", "bedrooms", "age", "transit"]]
X = sm.add_constant(X)          # adds intercept column "const"
y = df["price"]

model   = sm.OLS(y, X)
results = model.fit()           # OLS results object — all diagnostics live here

print(results.summary())

# ─────────────────────────────────────────────────────────────
# STEP 3 ▸ Extract diagnostics from the results object
# ─────────────────────────────────────────────────────────────

# results.fittedvalues  → Series of ŷ (predicted values for each obs)
# results.resid         → Series of e = y - ŷ (raw residuals)
# These are aligned by index — safe to combine directly into a DataFrame

y_hat     = results.fittedvalues          # ŷ: predicted prices
residuals = results.resid                 # e = y − ŷ

# Standardise residuals for outlier detection
resid_std  = residuals.std()
resid_mean = residuals.mean()

# Boolean mask: True when |e| > 2σ  (outlier threshold)
# np.abs gives element-wise absolute value; comparison yields a boolean Series
is_outlier = np.abs(residuals - resid_mean) > 2 * resid_std

# Compute RMSE
rmse = np.sqrt((residuals ** 2).mean())

# Build tidy DataFrame for Plotly
plot_df = pd.DataFrame({
    "y_hat":      y_hat,
    "residual":   residuals,
    "is_outlier": is_outlier,
    # Human-readable label for hover cards
    "category":   np.where(is_outlier, "Outlier (|e| > 2σ)", "Normal"),
    # Per-observation property labels
    "sqft":       df["sqft"].values,
    "bedrooms":   df["bedrooms"].values,
    "age":        df["age"].values,
}).reset_index(drop=True)

# ─────────────────────────────────────────────────────────────
# STEP 4 ▸ Build the forensics dashboard (3-panel layout)
# ─────────────────────────────────────────────────────────────

# Color palette — dark analytical theme
BACKGROUND   = "#0D1117"
PANEL        = "#161B22"
GRID         = "#21262D"
TEXT_PRIMARY = "#E6EDF3"
TEXT_DIM     = "#8B949E"
ACCENT_BLUE  = "#58A6FF"
CRIMSON      = "#DC143C"     # outlier color — stark, unambiguous
NORMAL_DOT   = "#238BEA"     # inlier color — calm blue

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "① Residuals vs. Fitted  (Heteroscedasticity Detector)",
        "② Residual Distribution  (Normality Check)",
        "③ Scale–Location Plot  (Spread Uniformity)",
        "④ Residuals Over Index  (Structural Break Scan)",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

# ── Shared scatter kwargs ──────────────────────────────────────
def scatter_kwargs(df_sub, color, name, symbol="circle", size=6):
    return dict(
        x=df_sub["y_hat"],
        y=df_sub["residual"],
        mode="markers",
        name=name,
        marker=dict(color=color, size=size, symbol=symbol,
                    line=dict(width=0.4, color="#0D1117")),
        customdata=np.stack([
            df_sub["sqft"], df_sub["bedrooms"], df_sub["age"]
        ], axis=1),
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Fitted ŷ: $%{x:,.0f}<br>"
            "Residual e: $%{y:,.0f}<br>"
            "Sqft: %{customdata[0]:.0f} | "
            "Beds: %{customdata[1]} | "
            "Age: %{customdata[2]:.1f}y"
            "<extra></extra>"
        ),
    )

normal_df  = plot_df[~plot_df["is_outlier"]]
outlier_df = plot_df[ plot_df["is_outlier"]]

# ── Panel 1 ▸ Residuals vs. Fitted ────────────────────────────
fig.add_trace(go.Scatter(**scatter_kwargs(normal_df,  NORMAL_DOT, "Normal")),  row=1, col=1)
fig.add_trace(go.Scatter(**scatter_kwargs(outlier_df, CRIMSON,    "Outlier",
                                          symbol="diamond", size=9)),           row=1, col=1)

# Zero-line reference — any non-random pattern here signals model misspecification
fig.add_hline(y=0, line=dict(color="#F0E68C", width=1.5, dash="dash"), row=1, col=1)

# 2σ envelope lines
fig.add_hline(y= 2 * resid_std, line=dict(color=CRIMSON, width=0.8, dash="dot"), row=1, col=1)
fig.add_hline(y=-2 * resid_std, line=dict(color=CRIMSON, width=0.8, dash="dot"), row=1, col=1)

# ── Panel 2 ▸ Residual Histogram ──────────────────────────────
fig.add_trace(
    go.Histogram(
        x=plot_df["residual"],
        nbinsx=40,
        name="Residual freq.",
        marker=dict(color=ACCENT_BLUE, opacity=0.75,
                    line=dict(color=BACKGROUND, width=0.5)),
        showlegend=False,
    ),
    row=1, col=2,
)
# Overlay a normal curve for visual reference
x_norm = np.linspace(plot_df["residual"].min(), plot_df["residual"].max(), 300)
y_norm = (n * (plot_df["residual"].max() - plot_df["residual"].min()) / 40
          * (1 / (resid_std * np.sqrt(2 * np.pi)))
          * np.exp(-0.5 * ((x_norm - resid_mean) / resid_std) ** 2))
fig.add_trace(
    go.Scatter(x=x_norm, y=y_norm, mode="lines",
               line=dict(color="#F0E68C", width=2),
               name="Normal curve", showlegend=False),
    row=1, col=2,
)
fig.add_vline(x=0, line=dict(color="#F0E68C", width=1.2, dash="dash"), row=1, col=2)

# ── Panel 3 ▸ Scale–Location (√|e| vs ŷ) ─────────────────────
# √|e| should be flat if variance is homoscedastic; an upward slope = heteroscedasticity
sqrt_abs_resid = np.sqrt(np.abs(plot_df["residual"]))

fig.add_trace(
    go.Scatter(
        x=plot_df["y_hat"],
        y=sqrt_abs_resid,
        mode="markers",
        name="√|e|",
        showlegend=False,
        marker=dict(color=np.where(plot_df["is_outlier"], CRIMSON, NORMAL_DOT),
                    size=5, opacity=0.7,
                    line=dict(width=0.3, color=BACKGROUND)),
        hovertemplate="ŷ: $%{x:,.0f}<br>√|e|: %{y:.2f}<extra></extra>",
    ),
    row=2, col=1,
)
# LOWESS smoothing line — slope reveals variance trend
from statsmodels.nonparametric.smoothers_lowess import lowess
smoothed = lowess(sqrt_abs_resid, plot_df["y_hat"], frac=0.3, return_sorted=True)
fig.add_trace(
    go.Scatter(x=smoothed[:, 0], y=smoothed[:, 1],
               mode="lines", line=dict(color="#FF7F50", width=2.5),
               name="LOWESS trend", showlegend=False),
    row=2, col=1,
)

# ── Panel 4 ▸ Residuals Over Index (structural break scan) ────
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df["residual"],
        mode="markers+lines",
        name="Residual series",
        showlegend=False,
        line=dict(color=ACCENT_BLUE, width=0.6),
        marker=dict(color=np.where(plot_df["is_outlier"], CRIMSON, ACCENT_BLUE),
                    size=np.where(plot_df["is_outlier"], 8, 4),
                    opacity=0.8),
        hovertemplate="Obs %{x}<br>e: $%{y:,.0f}<extra></extra>",
    ),
    row=2, col=2,
)
fig.add_hline(y=0,              line=dict(color="#F0E68C", width=1.5, dash="dash"), row=2, col=2)
fig.add_hline(y= 2*resid_std,  line=dict(color=CRIMSON,   width=0.8, dash="dot"),  row=2, col=2)
fig.add_hline(y=-2*resid_std,  line=dict(color=CRIMSON,   width=0.8, dash="dot"),  row=2, col=2)

# ─────────────────────────────────────────────────────────────
# STEP 5 ▸ Global layout styling
# ─────────────────────────────────────────────────────────────

n_outliers   = is_outlier.sum()
outlier_pct  = 100 * n_outliers / n

fig.update_layout(
    title=dict(
        text=(
            f"<b>RESIDUAL FORENSICS DASHBOARD</b>  ·  Hedonic Pricing OLS<br>"
            f"<sup>n={n:,} obs  |  RMSE = ${rmse:,.0f}  |  "
            f"Outliers (|e|>2σ): {n_outliers} ({outlier_pct:.1f}%)</sup>"
        ),
        x=0.5, xanchor="center",
        font=dict(size=18, color=TEXT_PRIMARY, family="Courier New, monospace"),
    ),
    paper_bgcolor=BACKGROUND,
    plot_bgcolor=PANEL,
    font=dict(color=TEXT_DIM, family="Courier New, monospace", size=11),
    legend=dict(
        bgcolor=PANEL, bordercolor=GRID, borderwidth=1,
        font=dict(color=TEXT_PRIMARY, size=11),
        x=0.01, y=0.99,
    ),
    height=780,
    margin=dict(l=60, r=40, t=100, b=60),
)

# Uniform axis styling across all panels
for axis in fig.layout:
    if axis.startswith("xaxis") or axis.startswith("yaxis"):
        fig.layout[axis].update(
            gridcolor=GRID, zeroline=False,
            showline=True, linecolor=GRID,
            tickfont=dict(color=TEXT_DIM, size=10),
            title_font=dict(color=TEXT_DIM),
        )

# Per-panel axis labels
fig.update_xaxes(title_text="Fitted Values ŷ  ($)",   row=1, col=1)
fig.update_yaxes(title_text="Residual e = y − ŷ  ($)", row=1, col=1)
fig.update_xaxes(title_text="Residual e  ($)",         row=1, col=2)
fig.update_yaxes(title_text="Frequency",               row=1, col=2)
fig.update_xaxes(title_text="Fitted Values ŷ  ($)",   row=2, col=1)
fig.update_yaxes(title_text="√|Residual|",             row=2, col=1)
fig.update_xaxes(title_text="Observation Index",       row=2, col=2)
fig.update_yaxes(title_text="Residual e  ($)",         row=2, col=2)

# Subplot title styling
for annotation in fig.layout.annotations:
    annotation.font.update(color=TEXT_PRIMARY, size=12)

fig.show()

# ─────────────────────────────────────────────────────────────
# STEP 6 ▸ Console summary
# ─────────────────────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"  RESIDUAL FORENSICS  ·  OLS Hedonic Pricing Model")
print(f"{'─'*55}")
print(f"  Observations   : {n:,}")
print(f"  R²             : {results.rsquared:.4f}")
print(f"  Adj. R²        : {results.rsquared_adj:.4f}")
print(f"  RMSE           : ${rmse:,.2f}")
print(f"  Residual Mean  : ${resid_mean:,.2f}  (should ≈ 0)")
print(f"  Residual Std   : ${resid_std:,.2f}")
print(f"  2σ threshold   : ±${2*resid_std:,.2f}")
print(f"  Outliers       : {n_outliers} ({outlier_pct:.1f}% of sample)")
print(f"{'─'*55}\n")

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.857
Model:                            OLS   Adj. R-squared:                  0.856
Method:                 Least Squares   F-statistic:                     742.2
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          1.60e-207
Time:                        20:10:05   Log-Likelihood:                -5929.9
No. Observations:                 500   AIC:                         1.187e+04
Df Residuals:                     495   BIC:                         1.189e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        5.03e+04   8873.021      5.669      0.0


───────────────────────────────────────────────────────
  RESIDUAL FORENSICS  ·  OLS Hedonic Pricing Model
───────────────────────────────────────────────────────
  Observations   : 500
  R²             : 0.8571
  Adj. R²        : 0.8559
  RMSE           : $34,229.21
  Residual Mean  : $0.00  (should ≈ 0)
  Residual Std   : $34,263.49
  2σ threshold   : ±$68,526.98
  Outliers       : 24 (4.8% of sample)
───────────────────────────────────────────────────────

